# Step 4: 常见报错 cheatsheet — 诊断映射 + 修复闭环

**目标**：把"看到报错"和"根因 + 属于哪类契约违反 + 怎么修"连起来——建立诊断决策树，把 vLLM 部署量化模型的常见报错（`No compatible kernel found` / AWQ `auto_awq` / `zero_point` / NCCL hang / OOM）映射到 s1 §5 的三条件（scheme/布局/kernel 路径）或环境/版本类，给出修复动作。

**对应 OUTLINE 课时**：4.4 常见报错 cheatsheet（~45 分钟）。

> **诊断锚点**：s1 的「三条件 + 边界外三条」是本节所有诊断的归因框架。每个报错最终都落到「标准 scheme / 权重布局 / kernel 路径」三条件之一，或环境/版本问题。


## 学完应能讲清（学完本节应能口头回答）

1. `No compatible kernel found` 的根因有哪几类（compressed-tensors 版本不匹配 / 架构没补 quantized kernel / 权重布局不符）？分别对应 §5 三条件的哪个？怎么修？（提示：版本→环境类升级；架构没补→条件③走 transformers fallback；布局不符→条件②重新量化）
2. AWQ 报 `--quantization awq` 错（应是 `auto_awq`）/ `zero_point` 错（AutoAWQ 须 `zero_point=False`）分别怎么修？（提示：前者改 flag；后者重新量化，Marlin kernel 要求 zero_point=False）
3. TP 启动 hang / rank timeout 怎么排查 NCCL（`pip show nvidia-nccl-cu12` / `--disable-custom-all-reduce` / `NCCL_DEBUG=INFO`）？（提示：版本查证→禁自定义 all-reduce 回退 NCCL→DEBUG 看通信死锁细节）
4. OOM at profile run 多半是什么（KV 太大不是权重）？怎么调（降 `--gpu-memory-utilization` 或 `--max-model-len`）？（提示：profile run 探 KV 块数时 OOM=KV 配置太激进，不是模型放不下；调小 utilization/max-len）


In [ ]:
%%capture
import pathlib, os, json
import ipytest
ipytest.autoconfig()


In [ ]:
# Setup cell（cwd 无关路径解析）。M4 跨模块读 M2/M3 7B 量化产物。
import pathlib

def _find_module_root(start):
    p = pathlib.Path(start).resolve()
    for cand in [p, *p.parents]:
        if (cand / "scripts").is_dir() and (cand / "steps").is_dir():
            return cand
    raise RuntimeError("找不到模块根（含 scripts/ + steps/ 的目录）")

MODULE_ROOT   = _find_module_root(pathlib.Path.cwd())
MODEL_DIR      = MODULE_ROOT / "models" / "Qwen2.5-7B-Instruct"
TINY_MODEL_DIR = MODULE_ROOT / "models" / "Qwen2.5-0.5B-Instruct"
OUT_ROOT       = MODULE_ROOT / "out"; OUT_ROOT.mkdir(parents=True, exist_ok=True)
REPO_COURSE = MODULE_ROOT.parent
M2_OUT = REPO_COURSE / "m2-quant-pipeline" / "out"
print("MODULE_ROOT =", MODULE_ROOT)


## 原理：报错归因到 §5 三条件 + 环境/版本

s4 的所有诊断最终落到 **5 个归因类**之一（前 3 个是 s1 §5 的三条件违反，后 2 个是环境/版本与 TP 通信）：

| 归因类 | 对应 §5 | 典型报错串 | 修复方向 |
|---|---|---|---|
| **scheme** | 条件① | `--quantization awq`（应是 `auto_awq`）/ unrecognized quant scheme | 改 flag / 用标准 scheme |
| **布局** | 条件② | `zero_point` shape error / group_size not divisible | 重新量化（AWQ zero_point=False / group_size 整除）|
| **kernel路径** | 条件③ | `No compatible kernel found` | 走 `--model-impl transformers` fallback / 换已适配架构 |
| **环境/版本** | — | `undefined symbol` / ImportError / CUDA version | `uv pip install -U compressed-tensors vllm` / 核对 CUDA |
| **NCCL/TP** | — | rank timeout / all-reduce error | `--disable-custom-all-reduce` / `NCCL_DEBUG=INFO` / spawn |

**高频报错根因速记**：
- `No compatible kernel found`：多数是**条件③**（架构没补 quantized kernel，尤其新架构只实现 bf16）或**条件②**（权重布局不符）。先查架构 vLLM 是否支持该量化。**版本类**：偶尔也来自 compressed-tensors 版本与 vllm 不匹配——表现为该 kernel 在运行时未注册（而非加载即报 `undefined symbol`）；遇此先 `uv pip install -U compressed-tensors vllm` 排除环境/版本因素，再查架构/kernel 路径或权重布局。
- AWQ `--quantization awq`：**scheme 类**——遗留 AutoAWQ flag 是 `auto_awq`（下划线！），标准 compressed-tensors AWQ 不用传 flag（走 auto）。
- AWQ `zero_point` 错：**布局类**——Marlin kernel 要求 `zero_point=False`；AutoAWQ 默认 True 会触发，要重新量化。
- `undefined symbol`：**环境类**——CUDA 版本与 vLLM wheel 不匹配（如用 CUDA 12.1 跑需 12.8 的 vLLM），driver 要 ≥570。
- OOM at profile run：**环境类**——profile run 探 KV 块数时 OOM = KV 配置太激进（不是权重放不下），降 `--gpu-memory-utilization` 或 `--max-model-len`。
- NCCL rank timeout：**TP 类**——多卡通信失败，`NCCL_DEBUG=INFO` 看死锁、`--disable-custom-all-reduce` 回退、`VLLM_WORKER_MULTIPROC_METHOD=spawn` 解 fork 问题。

## 亲手摸一摸：典型报错文本样本

看 OUTLINE 4.4 表里的典型报错串长什么样——为诊断填空打底（看特征词怎么映射到归因类）。


In [ ]:
# 摸一摸：典型报错文本样本（看特征词——归因规则留给你在 diagnose_error 里归纳，这里不预写答案）
SAMPLE_ERRORS = [
    "ValueError: No compatible kernel found for shape ...",
    "ValueError: --quantization awq is not a valid choice (try auto_awq)",
    "RuntimeError: zero_point must be False for Marlin kernel",
    "ImportError: libnccl.so: undefined symbol: ncclCommSplit",
    "RuntimeError: CUDA out of memory. Tried to allocate ... during profiling",
    "RuntimeError: NCCL communicator was initialized ... rank 1 timeout",
    "ValueError: group_size 127 is not divisible by hidden_size",
]
print("=== 典型报错样本（看每个串里的特征词，想想它属于哪类契约违反）===")
for e in SAMPLE_ERRORS:
    print(" -", e[:80])
print("\n（提示：特征词 → 归因类的映射，由你在下面的 diagnose_error 填空里归纳；")
print(" 参考 s1 §5 的三条件 + 环境/版本 + NCCL/TP 两类，不要直接在下方查表抄答案。）")

## 本步填空（2 个）

1. **`diagnose_error(error_msg)`**（判断型）— 给报错文本，返回 `(root_cause, category)`，`category` 映射到 §5 三条件之一（`scheme`/`布局`/`kernel路径`）或 `环境/版本`。**为什么这么设计（填前先想）**：诊断/判断型——把"看到报错"和"根因 + 属于哪类契约违反"连起来，是 4.4 最高频的工程能力。
2. **`recommend_fix(symptom)`** — 症状（报错文本或 `(root_cause, category)` 元组）→ 修复命令/动作。**为什么这么设计**：从诊断到修复的闭环；速查表逻辑化。


In [ ]:
def diagnose_error(error_msg):
    """给报错文本，返回 (root_cause: str, category: str)。
    category 取值：'scheme' | '布局' | 'kernel路径' | '环境/版本'（或 '未知'）。

    为什么这么设计（填前先想）：
    - 诊断型——每个报错都归到 §5 三条件之一或环境/版本。你按特征词做模式匹配，
      把"报错串"翻译成"契约违反类别 + 根因"，这就是 4.4 的核心工程能力。
    - 本模块部署 SmoothQuant W8A8 INT8，所以**布局类**报错重点关注 INT8 相关：
      group_size 整除、对称 per-channel/per-token 等；zero_point 是其他 scheme（如 AWQ）的布局约束，
      也归布局类但不是本模块主线。**kernel路径**类（No compatible kernel）对 W8A8 INT8 同样适用
      （架构没补 INT8 quantized kernel）。
    - 归因规则（按特征词，注意顺序：先环境/版本、再 kernel、再布局、再 scheme）：
        * ImportError/undefined symbol/libnccl    -> '环境/版本'（版本不匹配/CUDA 错配）
        * CUDA out of memory / profiling          -> '环境/版本'（KV 太大非权重）
        * NCCL/rank/timeout/all-reduce            -> '环境/版本'（TP 通信）
        * 'No compatible kernel' / 'kernel not found' -> 'kernel路径'（条件③）
        * zero_point / group_size / not divisible / shape -> '布局'（条件②）
        * '--quantization awq'（无 auto_awq）/ unrecognized quant -> 'scheme'（条件①）
    - 都不匹配 -> ('未识别...', '未知')。

    返回：(root_cause_human_readable, category)。
    """
    # TODO: error_msg 转小写后按上述规则逐条 if 匹配，返回 (root_cause, category)。
    #   root_cause 要写清「对应哪个条件不满足 / 具体原因」（如 'W8A8 INT8 kernel 路径缺失（条件③）'）。
    raise NotImplementedError

In [ ]:
%%ipytest -qq
# L1 测试（diagnose_error）——填完 diagnose_error 立即单独跑此 cell 验证（不依赖 recommend_fix）。

def test_diagnose_no_compatible_kernel():
    root, cat = diagnose_error("ValueError: No compatible kernel found for shape (1,4096)")
    assert cat == "kernel路径", cat
    assert "条件③" in root or "kernel" in root.lower()

def test_diagnose_awq_flag():
    root, cat = diagnose_error("ValueError: --quantization awq is not valid (try auto_awq)")
    assert cat == "scheme"
    assert "auto_awq" in root

def test_diagnose_zero_point():
    root, cat = diagnose_error("RuntimeError: zero_point must be False for Marlin kernel")
    assert cat == "布局"
    assert "条件②" in root or "zero_point" in root.lower() or "布局" in root

def test_diagnose_group_size():
    # W8A8 INT8 也可能触发布局类：group_size 整除
    root, cat = diagnose_error("ValueError: group_size 127 not divisible by hidden_size")
    assert cat == "布局"

def test_diagnose_undefined_symbol():
    root, cat = diagnose_error("ImportError: libnccl.so: undefined symbol: ncclCommSplit")
    assert cat == "环境/版本"

def test_diagnose_oom_profiling():
    root, cat = diagnose_error("RuntimeError: CUDA out of memory during profiling run")
    assert cat == "环境/版本"
    assert "KV" in root or "profiling" in root.lower() or "memory" in root.lower()

def test_diagnose_nccl_timeout():
    root, cat = diagnose_error("RuntimeError: NCCL rank 1 timeout after 600s")
    assert cat == "环境/版本"

In [ ]:
def recommend_fix(symptom):
    """症状 -> 修复命令/动作字符串。

    参数 symptom：
    - (root_cause, category) 元组：直接查修复表（**L1 测试只测这条路径，不依赖 diagnose_error**）
    - str（报错文本）：先调 diagnose_error 归因，再查修复表（**整合路径，放 L2 验证；L1 不测**）

    为什么这么设计（填前先想）：
    - 从诊断到修复的闭环——归因后立刻给出可执行动作（命令/参数调整），是 cheatsheet 的交付。
    - **测试解耦**：本填空的 L1 只验「结构化输入 (root,cat) -> 修复动作」这条纯查表路径，
      与 diagnose_error 是否已实现无关（diagnose_error 是另一个独立填空，L1 测试不依赖它）。
      str -> diagnose -> 查表 的整合路径在 L2 验（那时两个填空都填好了）。
    - 修复表按 category 分组：
        scheme     -> 改 flag（awq->auto_awq）/ 用标准 compressed-tensors scheme
        布局       -> 重新量化（W8A8/INT8 group_size 整除 hidden_size；其他 scheme 布局约束见 s1 §5）
        kernel路径 -> --model-impl transformers fallback 或换已适配架构
        环境/版本  -> 按根因细分：升级 compressed-tensors/vllm；降 gpu-memory-utilization/max-model-len；NCCL 三件套排查
    - 注意：symptom 是字符串时复用 diagnose_error（不重复归因逻辑）；元组时直接用其 category。

    返回：修复动作字符串。
    """
    # TODO:
    #   1) 若 symptom 是 str -> root,cat = diagnose_error(symptom)；否则 root,cat = symptom
    #   2) 按 cat 查上面的修复表（dict）。环境/版本 类要按 root 细分（dict 内 dict 或根因关键词）。
    #   3) 返回匹配的修复字符串。
    raise NotImplementedError

In [ ]:
%%ipytest -qq
# L1 测试（recommend_fix）——只测「结构化输入 (root,cat) 元组 -> 修复动作」纯查表路径。
# 故意只用元组输入（不传 str），这样本 cell 不依赖 diagnose_error 是否已实现（测试解耦）。
# str -> diagnose -> 查表 的整合路径在 L2 验（两个填空都填好后再跑）。

def test_recommend_fix_scheme():
    fix = recommend_fix(("--quantization awq 不是合法 scheme（条件①）", "scheme"))
    assert "auto_awq" in fix

def test_recommend_fix_layout_zero_point():
    fix = recommend_fix(("zero_point 布局不符（条件②）", "布局"))
    assert "zero_point=False" in fix or "重新量化" in fix

def test_recommend_fix_layout_group_size():
    fix = recommend_fix(("W8A8 group_size 不整除 hidden_size（条件②）", "布局"))
    assert "group_size" in fix or "整除" in fix or "重新量化" in fix

def test_recommend_fix_kernel_path():
    fix = recommend_fix(("架构未补 INT8 quantized kernel（条件③）", "kernel路径"))
    assert "model-impl" in fix or "fallback" in fix.lower() or "架构" in fix

def test_recommend_fix_env_version_oom():
    fix = recommend_fix(("profile run 时 OOM（KV 配置太激进，环境类）", "环境/版本"))
    assert "gpu-memory-utilization" in fix or "max-model-len" in fix

def test_recommend_fix_env_version_nccl():
    fix = recommend_fix(("NCCL rank timeout（TP 通信，环境类）", "环境/版本"))
    assert "disable-custom-all-reduce" in fix or "NCCL_DEBUG" in fix or "spawn" in fix

def test_recommend_fix_env_version_undefined_symbol():
    fix = recommend_fix(("undefined symbol（CUDA 版本错配，环境类）", "环境/版本"))
    assert "compressed-tensors" in fix or "vllm" in fix or "CUDA" in fix or "升级" in fix

## L2（CPU）：报错诊断 + 修复闭环（喂文本片段，纯逻辑）

L2 验 `diagnose_error` 对各典型报错的归因 + `recommend_fix` 闭环（CPU 可跑，喂报错文本片段）。


In [ ]:
## L2：对典型报错跑诊断 -> 修复闭环
print("=== L2：典型报错 -> 归因 -> 修复 ===")
for err in SAMPLE_ERRORS:
    root, cat = diagnose_error(err)
    fix = recommend_fix((root, cat))
    print("\n[报错] %s" % err[:70])
    print("  归因: [%s] %s" % (cat, root))
    print("  修复: %s" % fix)
# 验闭环：AWQ flag 错 -> 修 auto_awq
assert "auto_awq" in recommend_fix("--quantization awq invalid")
# 验闭环：kernel 不兼容 -> 走 fallback
assert "model-impl" in recommend_fix("No compatible kernel found") or "架构" in recommend_fix("No compatible kernel found")
print("\nL2 通过：所有典型报错正确归因到 §5 三条件/环境类，并给出可执行修复（诊断-修复闭环成立）。")


## L3（可选）：触发一个真报错并诊断

L3 可选：故意用一个错的 flag（如 `--quantization awq`）起 0.5B 服务，捕获报错文本，跑 `diagnose_error` 验诊断逻辑对真报错有效。纯逻辑已够（s4 的诊断/修复闭环 L1+L2 已验），L3 留作演示。

> **L3 双守卫**：`torch.cuda.is_available() and not os.environ.get('SKIP_L3')`——reviewer 执行验证设 `SKIP_L3=1` 跳过；真人跑时不设，L3 实证。


In [ ]:
import torch, os, subprocess, signal, time

def run_l3_trigger_error():
    # 故意用错 flag 起 0.5B 服务，捕获报错文本
    cmd = ("vllm serve %s --quantization awq --enforce-eager "
           "--gpu-memory-utilization 0.3" % str(TINY_MODEL_DIR))
    print("[L3] 故意用错 flag 起服务（捕获报错）：", cmd)
    try:
        proc = subprocess.Popen(cmd, shell=True, preexec_fn=os.setsid,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        try:
            out, _ = proc.communicate(timeout=60)
        except subprocess.TimeoutExpired:
            os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
            out = ""
        err_snippet = out[-800:] if out else "(无输出/超时)"
        print("\n[L3] 捕获到报错片段：")
        print("  " + err_snippet.replace("\n", "\n  "))
        root, cat = diagnose_error(err_snippet)
        fix = recommend_fix((root, cat))
        print("\n[L3] diagnose -> [%s] %s" % (cat, root))
        print("[L3] fix ->", fix)
    except Exception as e:
        print("[L3] 跳过（无法触发/捕获报错）:", e)

if torch.cuda.is_available() and not os.environ.get('SKIP_L3'):
    run_l3_trigger_error()
else:
    print("跳过 L3：无 GPU 或 SKIP_L3=1（诊断逻辑 L1+L2 已验；真人跑时不设 SKIP_L3 可演示真报错）。")


## 产物检查：诊断-修复闭环 cheatsheet

本节的交付是一张可查的 cheatsheet：`报错特征词 -> 归因（§5 三条件/环境）-> 修复动作`。L2 跑通即证明这张表对 OUTLINE 4.4 的所有典型报错都成立。

**回顾诊断决策树**（背下来 = vLLM 部署量化模型 90% 报错能自诊）：
1. 报错含 `undefined symbol` / `ImportError` → **环境/版本**（升级 compressed-tensors/vllm，核 CUDA 12.8 driver≥570）。
2. 报错含 `No compatible kernel` → **kernel路径(条件③)**（架构没补 quantized kernel → `--model-impl transformers` fallback 或换架构）。
3. 报错含 `zero_point` / `group_size` / `shape` → **布局(条件②)**（重新量化，AWQ 须 zero_point=False，group_size 整除）。
4. 报错含 `--quantization awq` → **scheme(条件①)**（改 `auto_awq`，或标准 compressed-tensors 不传 flag）。
5. 报错含 `out of memory` + `profil` → **环境(KV 太大)**（降 `--gpu-memory-utilization` / `--max-model-len`）。
6. 报错含 `NCCL` / `rank` / `timeout` → **TP 通信**（`--disable-custom-all-reduce` / `NCCL_DEBUG=INFO` / spawn）。

下一步 s5 把整条 M2→M3→M4 链路串成端到端闭环。
